In [1]:
from tensorflow.keras import layers,models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [2]:
classes = ["Earthquake","Flood","Landslide","No-disaster","Wildfire"]

In [3]:

img_height,img_width = 128,128
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 30,
    zoom_range = 0.3,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    horizontal_flip = True,
    brightness_range=[0.7,1.3],
    validation_split = 0.2,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)
base_model.trainable = False

In [4]:

train_datagen = train_datagen.flow_from_directory(
    'Disaster-multiclass/train',
    target_size=(img_height,img_width),
    class_mode = "categorical",
    batch_size = batch_size  
)

test_datagen = test_datagen.flow_from_directory(
    'Disaster-multiclass/test',
    target_size=(img_height,img_width),
    class_mode = "categorical",
    batch_size = batch_size  
)

val_datagen = val_datagen.flow_from_directory(
    'Disaster-multiclass/val',
    target_size=(img_height,img_width),
    class_mode = "categorical",
    batch_size = batch_size  
)



Found 8260 images belonging to 5 classes.
Found 780 images belonging to 5 classes.
Found 570 images belonging to 5 classes.


In [5]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(5, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
    
)

In [6]:
model.fit(train_datagen,validation_data = val_datagen,epochs = 10)

Epoch 1/10
259/259 [==============================] - 83s 314ms/step - loss: 1.1312 - accuracy: 0.5454 - val_loss: 1.0135 - val_accuracy: 0.5842
Epoch 2/10
259/259 [==============================] - 160s 618ms/step - loss: 0.9081 - accuracy: 0.6420 - val_loss: 0.9190 - val_accuracy: 0.6263
Epoch 3/10
259/259 [==============================] - 293s 1s/step - loss: 0.8588 - accuracy: 0.6534 - val_loss: 0.8623 - val_accuracy: 0.6474
Epoch 4/10
259/259 [==============================] - 221s 853ms/step - loss: 0.8398 - accuracy: 0.6668 - val_loss: 0.8424 - val_accuracy: 0.6596
Epoch 5/10
259/259 [==============================] - 81s 313ms/step - loss: 0.8035 - accuracy: 0.6817 - val_loss: 0.9233 - val_accuracy: 0.6544
Epoch 6/10
259/259 [==============================] - 80s 310ms/step - loss: 0.7952 - accuracy: 0.6851 - val_loss: 0.8872 - val_accuracy: 0.6526
Epoch 7/10
259/259 [==============================] - 82s 317ms/step - loss: 0.7916 - accuracy: 0.6889 - val_loss: 0.8344 - val_ac

In [7]:
test_loss, test_acc = model.evaluate(test_datagen)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

25/25 [==============================] - 6s 217ms/step - loss: 0.6852 - accuracy: 0.7372
Test Accuracy: 0.7371794581413269
Test Loss: 0.6852405071258545


In [8]:
print(train_datagen.class_indices)
print(test_datagen.class_indices)
print(val_datagen.class_indices)

{'Earthquake': 0, 'Flood': 1, 'Landslide': 2, 'No-disaster': 3, 'Wildfire': 4}
{'Earthquake': 0, 'Flood': 1, 'Landslide': 2, 'No-disaster': 3, 'Wildfire': 4}
{'Earthquake': 0, 'Flood': 1, 'Landslide': 2, 'No-disaster': 3, 'Wildfire': 4}


In [9]:
model.save("multi-Disaster_detector3.keras")

In [11]:
import tensorflow as tf
import numpy as np
import cv2
model = tf.keras.models.load_model("multi-Disaster_detector3.keras")
IMG_SIZE = 128

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    print("Prediction:", classes[class_index])

predict_image(r"C:\Users\User\Desktop\Project Disaster management\Dataset-2\test\non_disaster\06_04_0918.png")

1/1 [==============================] - 0s 446ms/step
Prediction: No-disaster


In [21]:
import os
from PIL import Image

dataset_path = 'Disaster-multiclass/trai'

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)
        
        try:
            img = Image.open(path)
            img.verify()
        except:
            print("Removing corrupted image:", path)
            os.remove(path)

print("Dataset cleaned successfully!")

Dataset cleaned successfully!
